# Naruto Arena RL Training

Use this notebook from Google Colab or the VS Code Colab extension with a GPU runtime. The first cell clones the training repo into the Colab VM, changes into that checkout, and runs everything from there.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/joaovicentedev/shinobi-arena-rl.git'
PROJECT_DIR = Path('/content/shinobi-arena-rl')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print('repo_exists=', PROJECT_DIR)

os.chdir(PROJECT_DIR)
subprocess.run(['git', 'pull', '--ff-only'], check=True)

print('working_dir=', Path.cwd())
print('has_pyproject=', Path('pyproject.toml').exists())
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

## Install Dependencies

In [ ]:
!python -m pip install -q uv
!uv sync --extra rl

## Check Runtime

In [ ]:
!nvidia-smi
!uv run --extra rl python - <<'PY'
import torch
from naruto_arena.rl.observation import observation_size, ROSTER, OBSERVATION_VERSION
print('cuda_available=', torch.cuda.is_available())
print('device=', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('roster_size=', len(ROSTER))
print('observation_version=', OBSERVATION_VERSION)
print('observation_size=', observation_size())
PY

## Train From Scratch

In [ ]:
MODEL_PATH = 'models/colab/naruto_actor_critic_transformer_colab.pt'

!uv run --extra rl python scripts/train_rl_pytorch.py \
  --model-arch transformer \
  --device auto \
  --opponent heuristic \
  --episodes 50000 \
  --batch-episodes 32 \
  --learning-rate 3e-4 \
  --log-interval 250 \
  --save-path {MODEL_PATH}

## Continue Training Against An Existing RL Model

In [ ]:
INIT_MODEL = 'models/colab/naruto_actor_critic_transformer_colab.pt'
NEXT_MODEL = 'models/colab/naruto_actor_critic_transformer_colab_vs_rl.pt'

!uv run --extra rl python scripts/train_rl_pytorch.py \
  --model-arch transformer \
  --device auto \
  --init-model-path {INIT_MODEL} \
  --opponent rl \
  --opponent-model-path {INIT_MODEL} \
  --episodes 50000 \
  --batch-episodes 32 \
  --learning-rate 1e-4 \
  --log-interval 250 \
  --save-path {NEXT_MODEL}

## Quick Evaluation

In [ ]:
!uv run --extra rl python scripts/tournament_rl.py \
  --model-path {MODEL_PATH} \
  --matches-per-pair 1 \
  --max-actions 300 \
  --report-path reports/colab_tournament.json

## Optional: Copy Checkpoints To Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/naruto-arena-models
# !cp -v models/colab/*.pt /content/drive/MyDrive/naruto-arena-models/